# roster.db 트러블슈팅 쿼리 모음

파이프라인 진행 상황 확인용. 위에서부터 순서대로 실행하면 됨.
(노트북은 repo 루트에서 열어야 `data/roster.db` 경로가 맞음)

In [ ]:
import sqlite3
from pathlib import Path

DB = Path("data/roster.db")
assert DB.exists(), f"{DB.resolve()} 없음 — import_roster.py 먼저 실행했는지, 노트북을 repo 루트에서 열었는지 확인"
conn = sqlite3.connect(DB)
conn.row_factory = sqlite3.Row

def q(sql, *args):
    """SELECT 실행 후 행을 dict로 출력. 반환값도 dict 리스트."""
    rows = [dict(r) for r in conn.execute(sql, args)]
    for r in rows:
        print(r)
    print(f"-- {len(rows)}행")
    return rows

In [ ]:
# 전체 현황: status별 인원 (NULL = 아직 split 안 돌린 파일의 인원)
q("SELECT status, COUNT(*) AS cnt FROM roster GROUP BY status");

In [ ]:
# 파일별 진행 현황: split/정제가 어디까지 됐는지 한눈에
q("""
SELECT source_file,
       COUNT(*) AS 인원,
       SUM(status = '정상') AS 정상,
       SUM(status = '이상') AS 이상,
       SUM(normalized_filename IS NOT NULL AND normalized_filename != '') AS 정제완료
FROM roster GROUP BY source_file ORDER BY source_file
""");

In [ ]:
# 이상 목록: 페이지 배정 확인 필요한 사람들 (split 결과 검수 대상)
q("SELECT seq, name, source_file, source_file_seq, page_start, page_end FROM roster WHERE status = '이상' ORDER BY source_file, source_file_seq");

In [ ]:
# 정상인데 아직 미정제: 다음 normalize_person_text.py 실행 시 처리될 대상
q("SELECT seq, name, cl_level, split_filename FROM roster WHERE status = '정상' AND (normalized_filename IS NULL OR normalized_filename = '')");

In [ ]:
# cl_level 값 검증: CL2/CL3/CL4 외의 값이 있으면 normalize가 KeyError로 죽음
q("SELECT DISTINCT cl_level FROM roster");

In [ ]:
# 동명이인 체크: 같은 pjt+part에 같은 이름이 있으면 split 파일이 덮어써짐
q("SELECT name, pjt, part, COUNT(*) AS cnt FROM roster GROUP BY name, pjt, part HAVING cnt > 1");

In [ ]:
# 특정 사람 검색 (이름 일부로)
q("SELECT * FROM roster WHERE name LIKE ?", "%안지유%");

In [ ]:
# merge_persons.py 실행 시 제외될 사람 (정제 미완료) 미리보기
q("SELECT seq, name, status, split_filename FROM roster WHERE normalized_filename IS NULL OR normalized_filename = ''");